### Displacement-Based Ritz-DQ Method for Plate Post-Buckling

This notebook implements a purely displacement-based Ritz-Differential Quadrature (Ritz-DQ) method to solve the large-displacement post-buckling behavior of composite/isotropic plates. It contrasts classic Yamaki I (straight) and Yamaki III (stress-free) boundary conditions.

#### Ritz-DQ vs. Analytical DQM

It is crucial to mathematically distinguish the Ritz-DQ method used here from the [Analytical DQM](PostBuckling-DQM-Analytical-Airy.ipynb) based on the Airy stress function. The analytical DQM uses the strong form and operates on the governing partial differential equations (PDEs), such as the Föppl-von Kármán equations. It discretizes the spatial domain using collocation points and approximates derivatives using weighted linear sums ($D^{(m)}$ matrices). Boundary conditions are enforced by directly modifying the discrete residual equations. The Ritz-DQ uses the weak form, or integrated form, or energy form; operating on the total potential energy functional ($\Pi$). 

The Ritz part approximates the continuous displacement fields ($u, v, w$) globally using a truncated series of hierarchical, kinematically admissible trial functions. The DQ part evaluates the resulting energy integrals numerically using Gauss-Legendre quadrature. Because polynomials integrate exactly under Gaussian quadrature, this eliminates the need to symbolically integrate massive polynomial expansions. The method optimizes the scalar coefficients of the trial functions to minimize the energy.

By adopting Ritz-DQ, we avoid formulating the Airy Stress function ($\Phi$) entirely and work purely with displacements.

In [5]:
import numpy as np
from scipy.special import legendre
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# Global Parameters
# ==========================================
E = 70e9       # Elastic Modulus (Pa)
nu = 0.3       # Poisson's ratio
h = 0.01       # Plate thickness (m)
a = 1.0        # Plate length (m)
b = 1.0        # Plate width (m)

D = (E * h**3) / (12 * (1 - nu**2))
P_cr = (4 * np.pi**2 * D) / (b**2)

# Classical Lamination Theory (CLT) Stiffnesses (Isotropic Case)
A11 = A22 = (E * h) / (1 - nu**2)
A12 = nu * A11
A66 = (E * h) / (2 * (1 + nu))

D11 = D22 = D
D12 = nu * D11
D66 = (E * h**3) / (24 * (1 + nu))

# Resolution Settings
N_terms = 8  # Number of hierarchical polynomial modes per direction
N_gauss = 20 # Number of Gauss-Legendre integration points

## 2. Von Kármán Kinematics & Potential Energy

The total potential energy of the system is $\Pi = U_{membrane} + U_{bending}$. To capture post-buckling geometric stiffening, the midplane strains must include the nonlinear von Kármán stretching terms:

$$ \varepsilon_x = u_{,x} + \frac{1}{2}w_{,x}^2, \quad \varepsilon_y = v_{,y} + \frac{1}{2}w_{,y}^2, \quad \gamma_{xy} = u_{,y} + v_{,x} + w_{,x}w_{,y} $$
$$ \kappa_x = -w_{,xx}, \quad \kappa_y = -w_{,yy}, \quad \kappa_{xy} = -2w_{,xy} $$

The internal strain energy is the integral of the stress resultants over the domain:
$$ U = \frac{1}{2} \int_{-a/2}^{a/2} \int_{-b/2}^{b/2} (N_x \varepsilon_x + N_y \varepsilon_y + N_{xy} \gamma_{xy} + M_x \kappa_x + M_y \kappa_y + M_{xy} \kappa_{xy}) \, dx \, dy $$

To solve this numerically, we map the coordinates $x,y$ to a normalized domain $\xi, \eta \in [-1, 1]$. The integral is exactly evaluated using Gauss-Legendre quadrature weights.

#### Hierarchical Basis Functions (Penalty-Free Boundary Engine)

Standard penalty methods enforce boundary conditions via artificial cohesive zone stiffnesses (e.g., $k_i \approx 10^{14}$). This severely ill-conditions the global stiffness matrix. 

We eliminate cohesive penalties entirely by using **Integrated Legendre Polynomials** (Bardell, 1989; Beslin, 1997). The standard Legendre polynomial $P_n(\xi)$ evaluates to $P_n(1) = 1$ and $P_n(-1) = (-1)^n$. By defining the *internal* modes as the difference of two Legendre polynomials:
$$ \phi_n(\xi) = P_{n-1}(\xi) - P_{n-3}(\xi) \quad \text{for } n \ge 3 $$
We guarantee that $\phi_n(\pm 1) = 0$ exactly. 

The boundary modes are defined as simple linear interpolants:
$$ \phi_1(\xi) = \frac{1-\xi}{2}, \quad \phi_2(\xi) = \frac{1+\xi}{2} $$

**Boundary Control Strategy:** To enforce a strict Dirichlet condition (e.g., $w=0$ at the edges), we simply truncate the series to exclude $\phi_1$ and $\phi_2$. The remaining trial functions inherently satisfy the geometric constraint perfectly.

In [6]:
# ==========================================
# Hierarchical Legendre Basis
# ==========================================
def P(n, x):
    """Standard Legendre Polynomial"""
    if n < 0: return np.zeros_like(x)
    return legendre(n)(x)

def phi(n, x):
    """Integrated Legendre Basis (Rodrigues' hierarchy)"""
    if n == 1: return (1 - x) / 2.0  # Boundary mode (xi = -1)
    if n == 2: return (1 + x) / 2.0  # Boundary mode (xi = +1)
    return P(n-1, x) - P(n-3, x)     # Internal bubble mode (zero at boundaries)

def dphi(n, x):
    """Analytical First Derivative"""
    if n == 1: return -0.5 * np.ones_like(x)
    if n == 2: return 0.5 * np.ones_like(x)
    return (2*n - 3) * P(n-2, x)

def d2phi(n, x):
    """Analytical Second Derivative"""
    if n == 1 or n == 2: return np.zeros_like(x)
    return (2*n - 3) * legendre(n-2).deriv()(x)

# Pre-calculate Gauss-Legendre quadrature points & 2D weights
xi_g, weights = np.polynomial.legendre.leggauss(N_gauss)
W_2D = np.outer(weights, weights)

def build_matrices(indices, func, grid):
    """Evaluates selected basis modes over the specified numerical grid."""
    return np.array([func(n, grid) for n in indices])

#### Modeling Yamaki Boundary Conditions

We investigate the classical straight versus warped edge formulations defined by Yamaki (1959).

##### Yamaki III: Stress-Free Unloaded Edges
The unloaded edges ($y = \pm b/2$) must have zero transverse stress ($N_y = 0$). To achieve this energetically, we leave the transverse displacement field $v(\xi, \eta)$ completely unconstrained at the boundaries. 
* **Implementation:** We include boundary modes $\phi_1$ and $\phi_2$ in the transverse ($\eta$) expansion. The optimizer naturally relaxes the edge to minimize energy, resulting in warping and $N_y 	o 0$.

##### Yamaki I: Straight Unloaded Edges
The unloaded edges must expand outwards due to the Poisson effect, but they must remain perfectly straight lines ($v_{,xx} = 0$).
* **Implementation:** We enforce $v=0$ dynamically by stripping $\phi_1$ and $\phi_2$ from the $\eta$ expansion. To allow uniform expansion, we introduce a single global scaling variable $V_0$. The total displacement becomes $v_{total} = v_{internal} + V_0 \eta$. The Ritz solver balances $V_0$ globally, ensuring a perfect straight edge without artificial cohesive zone constraints.

In [7]:
# ==========================================
# Energy Solver Engine
# ==========================================
def solve_postbuckling(condition):
    print(f"\n--- Initializing {condition} Tracker ---")
    
    # --- KINEMATIC BASIS SELECTION ---
    # u: Loaded edges locked to platens (u = +/- u0 at x = +/- a/2)
    idx_u_xi = range(3, N_terms + 1)  
    idx_u_eta = range(1, N_terms + 1) 
    
    # w: Simply supported on all 4 edges (w = 0)
    idx_w_xi = range(3, N_terms + 1)  
    idx_w_eta = range(3, N_terms + 1) 
    
    if condition == 'Yamaki_I':
        # Straight Edges: Internal modes only (v=0 at bndry) + V0 uniform scalar
        idx_v_xi = range(1, N_terms + 1)
        idx_v_eta = range(3, N_terms + 1) 
        use_V0 = True
    else: 
        # Yamaki III (Warped Edges): Allow full boundary modes to relax Ny to 0
        idx_v_xi = range(1, N_terms + 1)
        idx_v_eta = range(1, N_terms + 1)
        use_V0 = False
        
    len_u = len(idx_u_xi) * len(idx_u_eta)
    len_v = len(idx_v_xi) * len(idx_v_eta)
    len_w = len(idx_w_xi) * len(idx_w_eta)
    total_dofs = len_u + len_v + len_w + (1 if use_V0 else 0)

    # Pre-evaluate basis at Gauss Quadrature points
    Phi_u_xi = build_matrices(idx_u_xi, phi, xi_g);   dPhi_u_xi = build_matrices(idx_u_xi, dphi, xi_g)
    Phi_u_eta = build_matrices(idx_u_eta, phi, xi_g); dPhi_u_eta = build_matrices(idx_u_eta, dphi, xi_g)
    Phi_v_xi = build_matrices(idx_v_xi, phi, xi_g);   dPhi_v_xi = build_matrices(idx_v_xi, dphi, xi_g)
    Phi_v_eta = build_matrices(idx_v_eta, phi, xi_g); dPhi_v_eta = build_matrices(idx_v_eta, dphi, xi_g)
    Phi_w_xi = build_matrices(idx_w_xi, phi, xi_g);   dPhi_w_xi = build_matrices(idx_w_xi, dphi, xi_g); d2Phi_w_xi = build_matrices(idx_w_xi, d2phi, xi_g)
    Phi_w_eta = build_matrices(idx_w_eta, phi, xi_g); dPhi_w_eta = build_matrices(idx_w_eta, dphi, xi_g); d2Phi_w_eta = build_matrices(idx_w_eta, d2phi, xi_g)

    # --- PRECONDITIONER ---
    # Ritz optimizers fail on finite difference gradients if inputs are not O(1).
    # We scale DOFs by physical magnitudes: U/V ~ O(h^2/a), W ~ O(h)
    scale_vec = np.ones(total_dofs)
    scale_vec[0:len_u] = (h**2) / a
    if use_V0:
        scale_vec[len_u] = (h**2) / b
        scale_vec[len_u+1 : len_u+1+len_v] = (h**2) / b
        scale_vec[len_u+1+len_v :] = h
    else:
        scale_vec[len_u : len_u+len_v] = (h**2) / b
        scale_vec[len_u+len_v :] = h

    def decode_dofs(q):
        """Unpacks 1D optimizer array into physical 2D coefficient matrices."""
        U_mat = q[0:len_u].reshape((len(idx_u_xi), len(idx_u_eta)))
        offset = len_u
        if use_V0:
            V0 = q[offset]
            offset += 1
        else:
            V0 = 0.0
        V_mat = q[offset : offset+len_v].reshape((len(idx_v_xi), len(idx_v_eta)))
        offset += len_v
        W_mat = q[offset : offset+len_w].reshape((len(idx_w_xi), len(idx_w_eta)))
        return U_mat, V0, V_mat, W_mat

    def strain_energy(q_phys, u0_val):
        """Evaluates Total Potential Energy via numerical exact Gauss Quadrature."""
        U_mat, V0, V_mat, W_mat = decode_dofs(q_phys)
        
        # Kinematic Field Reconstruction
        u_x = (2/a) * (dPhi_u_xi.T @ U_mat @ Phi_u_eta - u0_val)  # Platen compression
        u_y = (2/b) * (Phi_u_xi.T @ U_mat @ dPhi_u_eta)
        
        v_x = (2/a) * (dPhi_v_xi.T @ V_mat @ Phi_v_eta)
        v_y = (2/b) * (Phi_v_xi.T @ V_mat @ dPhi_v_eta) + V0      # Yamaki I uniform expansion
        
        w_x = (2/a) * (dPhi_w_xi.T @ W_mat @ Phi_w_eta)
        w_y = (2/b) * (Phi_w_xi.T @ W_mat @ dPhi_w_eta)
        w_xx = (2/a)**2 * (d2Phi_w_xi.T @ W_mat @ Phi_w_eta)
        w_yy = (2/b)**2 * (Phi_w_xi.T @ W_mat @ d2Phi_w_eta)
        w_xy = (2/a)*(2/b) * (dPhi_w_xi.T @ W_mat @ dPhi_w_eta)
        
        # Nonlinear von Karman Strains & Curvatures
        ex, ey, gxy = u_x + 0.5*w_x**2, v_y + 0.5*w_y**2, u_y + v_x + w_x*w_y
        kx, ky, kxy = -w_xx, -w_yy, -2*w_xy
        
        # Constitutive Law
        Nx, Ny, Nxy = A11*ex + A12*ey, A12*ex + A22*ey, A66*gxy
        Mx, My, Mxy = D11*kx + D12*ky, D12*kx + D22*ky, D66*kxy
        
        # Integral (Joules)
        U_total = 0.5 * (Nx*ex + Ny*ey + Nxy*gxy + Mx*kx + My*ky + Mxy*kxy)
        return np.sum(U_total * W_2D) * (a * b / 4.0)

    def normalized_energy(q_opt, u0_val):
        return strain_energy(q_opt * scale_vec, u0_val) / D

    # --- PATH TRACKER ---
    u0_cr = (np.pi**2 * h**2 * a) / (6 * (1 - nu**2) * b**2)
    u0_steps = np.linspace(u0_cr * 1.05, 0.003, 15) # Start strictly post-bifurcation

    P_history, w_history = [], []
    final_q_phys = None
    
    q_opt_prev = None
    q_opt_curr = np.zeros(total_dofs)
    idx_w_start = len_u + (1 if use_V0 else 0) + len_v
    q_opt_curr[idx_w_start] = 0.1  # Initial fundamental symmetric buckle mode seed

    for idx, u0 in enumerate(u0_steps):
        # First-Order Secant Extrapolator (Stabilizes deep post-buckling)
        if idx < 2:
            q_guess = q_opt_curr.copy()
        else:
            q_guess = q_opt_curr + (q_opt_curr - q_opt_prev)

        # Utilizing Conjugate Gradient (CG) for robustness against FD noise.
        res = minimize(normalized_energy, q_guess, args=(u0,), method='CG', 
                       options={'gtol': 1e-4, 'eps': 1e-6})
        
        if not res.success:
            print(f"  [!] Minimizer failed at u0 = {u0:.5f} | {res.message}")
            break
            
        q_opt_prev, q_opt_curr = q_opt_curr.copy(), res.x.copy()
        final_q_phys = res.x * scale_vec
        _, _, _, W_mat = decode_dofs(final_q_phys)
        
        # Center deflection magnitude
        phi_w_c = np.array([phi(n, 0.0) for n in idx_w_xi])
        w_max = phi_w_c.T @ W_mat @ phi_w_c
        
        # Reaction Force Calculation via Virtual Work (Castigliano)
        du = 1e-6
        Force = (strain_energy(final_q_phys, u0+du) - strain_energy(final_q_phys, u0-du)) / (4*du)
        
        P_ratio, w_ratio = (Force / b) / P_cr, np.abs(w_max) / h
        P_history.append(P_ratio)
        w_history.append(w_ratio)
        print(f" u0 = {u0:.5f} | Converged P/P_cr = {P_ratio:.4f} | W_max/h = {w_ratio:.4f}")

    indices = (idx_u_xi, idx_u_eta, idx_v_xi, idx_v_eta, idx_w_xi, idx_w_eta, use_V0)
    return w_history, P_history, final_q_phys, indices, decode_dofs, u0_steps[-1]

#### Post-Processing & 3D Visualization

To analyze the behavior, we plot the maximum deflection ($W/h$) against the compressive load ($P/P_{cr}$) and compare it against the exact analytical benchmark derived by Levy (1942).

We also render the 3D surface mapping. The in-plane displacements ($u, v$) are scaled artificially in the 3D render to visually expose the fundamental difference between Yamaki I and Yamaki III boundaries.

In [ ]:
# ==========================================
# Reconstruction and Plotting Utilities
# ==========================================
def extract_3d_fields(q_phys, indices, decode_dofs, u0_final):
    idx_u_xi, idx_u_eta, idx_v_xi, idx_v_eta, idx_w_xi, idx_w_eta, use_V0 = indices
    U_mat, V0, V_mat, W_mat = decode_dofs(q_phys)
    
    # Generate high-resolution plotting grid (Not Gauss Points)
    N_plot = 40
    xi = np.linspace(-1, 1, N_plot)
    eta = np.linspace(-1, 1, N_plot)
    XI, ETA = np.meshgrid(xi, eta, indexing='ij')
    
    Phi_u_xi, Phi_u_eta = build_matrices(idx_u_xi, phi, xi), build_matrices(idx_u_eta, phi, eta)
    Phi_v_xi, Phi_v_eta = build_matrices(idx_v_xi, phi, xi), build_matrices(idx_v_eta, phi, eta)
    Phi_w_xi, Phi_w_eta = build_matrices(idx_w_xi, phi, xi), build_matrices(idx_w_eta, phi, eta)
    
    # Total Continuous Field Reconstruction
    U_field = (Phi_u_xi.T @ U_mat @ Phi_u_eta) - u0_final * XI
    V_field = (Phi_v_xi.T @ V_mat @ Phi_v_eta) + (V0 * ETA if use_V0 else 0)
    W_field = Phi_w_xi.T @ W_mat @ Phi_w_eta
    
    X_phys, Y_phys = XI * (a/2), ETA * (b/2)
    return X_phys, Y_phys, U_field, V_field, W_field

def render_3d_deformation(fields_I, fields_III):
    X_I, Y_I, U_I, V_I, W_I = fields_I
    X_III, Y_III, U_III, V_III, W_III = fields_III
    
    mag = 40 # In-plane Magnification Factor for visual clarity
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=(f'Yamaki I (Straight Edges)<br>In-Plane Magnification: {mag}x', 
                        f'Yamaki III (Warped Edges)<br>In-Plane Magnification: {mag}x')
    )
    
    fig.add_trace(
        go.Surface(x=X_I + mag*U_I, y=Y_I + mag*V_I, z=W_I, 
                   surfacecolor=V_I, colorscale='Plasma',
                   colorbar=dict(title='Transverse Disp (v)', x=0.45)), row=1, col=1
    )
    
    fig.add_trace(
        go.Surface(x=X_III + mag*U_III, y=Y_III + mag*V_III, z=W_III, 
                   surfacecolor=V_III, colorscale='Plasma',
                   colorbar=dict(title='Transverse Disp (v)', x=1.0),
                   cmin=np.min(V_III), cmax=np.max(V_III)), row=1, col=2
    )
    
    fig.update_layout(
        title_text='3D Plate Deformation: Straight vs Warped Boundaries',
        width=1200, height=650,
        scene=dict(xaxis=dict(range=[-0.55, 0.55]), yaxis=dict(range=[-0.55, 0.55]), zaxis=dict(range=[0, np.max(W_I)*1.2])),
        scene2=dict(xaxis=dict(range=[-0.55, 0.55]), yaxis=dict(range=[-0.55, 0.55]), zaxis=dict(range=[0, np.max(W_I)*1.2]))
    )
    fig.show()

# ==========================================
# Main Execution
# ==========================================
if __name__ == '__main__':
    w_I, P_I, q_I, idx_I, dec_I, u0_I = solve_postbuckling('Yamaki_I')
    w_III, P_III, q_III, idx_III, dec_III, u0_III = solve_postbuckling('Yamaki_III')
    
    fields_I = extract_3d_fields(q_I, idx_I, dec_I, u0_I)
    fields_III = extract_3d_fields(q_III, idx_III, dec_III, u0_III)
    
    render_3d_deformation(fields_I, fields_III)
    
    # 2D Validation
    levy_W = [0.0, 0.50, 1.00, 1.50, 2.00]
    levy_P = [1.0, 1.15, 1.55, 2.15, 3.00]
    
    plt.figure(figsize=(9, 7))
    plt.plot(levy_W, levy_P, 'ko', markersize=8, label='Levy Exact (Yamaki I Benchmark)')
    plt.plot(w_I, P_I, 'b-', linewidth=2.5, marker='s', label='Ritz-DQ Hierarchical (Yamaki I: Straight)')
    plt.plot(w_III, P_III, 'r--', linewidth=2.5, marker='^', label='Ritz-DQ Hierarchical (Yamaki III: Warping)')
    
    plt.title('Displacement-Based Ritz Post-Buckling (Penalty-Free Boundary Engine)', fontsize=14, pad=15)
    plt.xlabel('Normalized Maximum Deflection ($W_{max}/h$)', fontsize=12)
    plt.ylabel('Applied Compressive Load Ratio ($P/P_{cr}$)', fontsize=12)
    plt.xlim([0, 2.2])
    plt.ylim([1.0, 3.1])
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper left', fontsize=11)
    plt.tight_layout()
    plt.show()


--- Initializing Yamaki_I Tracker ---
 u0 = 0.00019 | Converged P/P_cr = 1.0135 | W_max/h = 0.3588
 u0 = 0.00039 | Converged P/P_cr = 1.5766 | W_max/h = 1.2671
 u0 = 0.00059 | Converged P/P_cr = 2.1018 | W_max/h = 1.7524
 u0 = 0.00079 | Converged P/P_cr = 2.6020 | W_max/h = 2.1076
 u0 = 0.00099 | Converged P/P_cr = 3.0797 | W_max/h = 2.3888
 u0 = 0.00119 | Converged P/P_cr = 3.5275 | W_max/h = 2.6196
 u0 = 0.00139 | Converged P/P_cr = 3.9527 | W_max/h = 2.8173
 u0 = 0.00159 | Converged P/P_cr = 4.3583 | W_max/h = 2.9896
 u0 = 0.00180 | Converged P/P_cr = 4.7478 | W_max/h = 3.1408
 u0 = 0.00200 | Converged P/P_cr = 5.1298 | W_max/h = 3.2828
 u0 = 0.00220 | Converged P/P_cr = 5.5021 | W_max/h = 3.4152
 u0 = 0.00240 | Converged P/P_cr = 5.8675 | W_max/h = 3.5406
 u0 = 0.00260 | Converged P/P_cr = 6.2240 | W_max/h = 3.6581
 u0 = 0.00280 | Converged P/P_cr = 6.5779 | W_max/h = 3.7713
 u0 = 0.00300 | Converged P/P_cr = 6.9280 | W_max/h = 3.8811

--- Initializing Yamaki_III Tracker ---
 u0 =